In [ ]:
# pip install ollama

In [ ]:
# Para tratamiento de datos
import pandas as pd
import numpy as np
import re #para llamar a Expresiones Regulares y estandarizar el nombre de las columnas.
import time
import requests

# Para visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns

# Para poder visualizar todas las columnas de los DataFrames
pd.set_option('display.max_columns', None) 

# Trabajar con el sistema operativo y variables de entorno
import os 
from dotenv import load_dotenv
load_dotenv() #carga las variables del entorno .env; devuelve un true o false
import ollama
MODELO = 'gemma3:4b' 

# Conexión con MySQL
import mysql.connector
from mysql.connector import Error

# Gestión de los warnings
import warnings
warnings.filterwarnings("ignore")



In [ ]:
# Cargamos las variables de entorno del archivo .env
load_dotenv()

# 1. Recuperamos la Key y el ID base
steam_key = os.getenv("steam_key")

# 2. Creamos la lista de los 21 IDs dinámicamente
lista_ids = []
for i in range(21):
    variable_name = f"steam_id_{i}"
    valor_id = os.getenv(variable_name)
    if valor_id:
        lista_ids.append(valor_id)

print(f"Se han cargado {len(lista_ids)} IDs desde el archivo .env")

In [ ]:
load_dotenv()
steam_key = os.getenv("steam_key")

# Cargamos los 21 IDs desde el .env
ids_proyecto = [os.getenv(f"steam_id_{i}") for i in range(21)]
ids_proyecto = [i for i in ids_proyecto if i] # Limpiamos valores None

registros_juegos = []

print(f"--- Iniciando construcción del Dataset ({len(ids_proyecto)} usuarios) ---")

for s_id in ids_proyecto:
    # Usamos HTTPS y especificamos el formato JSON explícitamente
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        'key': steam_key,
        'steamid': s_id,
        'format': 'json',
        'include_appinfo': 1,
        'include_played_free_games': 1
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            # Accedemos a la estructura de la respuesta
            juegos = data.get('response', {}).get('games', [])
            
            for juego in juegos:
                juego['user_id'] = s_id
                registros_juegos.append(juego)
            
            print(f"✅ ID {s_id}: {len(juegos)} juegos añadidos.")
        else:
            print(f"❌ ID {s_id}: Error HTTP {response.status_code} (Posible perfil privado)")
            
    except Exception as e:
        print(f"⚠️ ID {s_id}: Error de conexión o formato: {e}")
    
    time.sleep(0.5) # Pausa para evitar el baneo de la API

# Creación del DataFrame y guardado
if registros_juegos:
    df_steam = pd.DataFrame(registros_juegos)
    df_steam.to_csv("./datasets/dataset_steam.csv", index=False)
    print(f"\n¡ÉXITO! Dataset guardado con {len(df_steam)} filas.")
else:
    print("\nNo se pudo extraer información. Revisa tu steam_key.")

In [ ]:
steam_db = pd.read_csv("./datasets/dataset_steam.csv")

In [ ]:
def obtener_estadisticas_logros(api_key, steam_id, app_id):
    # Endpoint para logros de usuario
    url = "https://api.steampowered.com/ISteamUserStats/GetPlayerAchievements/v0001/"
    params = {'key': api_key, 'steamid': steam_id, 'appid': app_id}
    
    try:
        r = requests.get(url, params=params, timeout=5)
        if r.status_code == 200:
            data = r.json()
            achievements = data.get('playerstats', {}).get('achievements', [])
            total = len(achievements)
            ganados = sum(1 for a in achievements if a.get('achieved') == 1)
            return ganados, total
    except:
        return 0, 0
    return 0, 0

# 3. Procesar una muestra significativa (ej. los 100 registros más jugados)
# Usamos 'user_id' que es como se guardó en tu CSV anterior
df_sample = steam_db.sort_values(by='playtime_forever', ascending=False).head(100).copy()

logros_data = []
print("Enriqueciendo datos con logros... (esto tardará un poco)")

for idx, row in df_sample.iterrows():
    # CAMBIO CLAVE: Usamos 'user_id' para evitar el KeyError de tu imagen
    ganados, totales = obtener_estadisticas_logros(steam_key, row['user_id'], row['appid'])
    logros_data.append({
        'achieved': ganados,
        'total_achievements': totales,
        'completion_percentage': (ganados / totales * 100) if totales > 0 else 0
    })
    time.sleep(0.2) # Pausa de cortesía para la API

# 4. Unir y guardar el dataset final
df_extra = pd.DataFrame(logros_data)
df_final = pd.concat([df_sample.reset_index(drop=True), df_extra], axis=1)

df_final.to_csv("./datasets/dataset_steam_final.csv", index=False)
print("¡Hecho! Dataset final guardado.")

In [ ]:
steam_db

In [ ]:
steam_db_final = pd.read_csv("./datasets/dataset_steam_final.csv")

In [ ]:
steam_db_final.head(10)

# DS Maestro de juegos

In [ ]:
data = {
    'appid': steam_db_final["appid"].unique(), 
    'name': steam_db_final["name"].unique()
}
df_juegos_maestro = pd.DataFrame(data)

# Consultamos la info en la API de Steam.
def obtener_info_juego(appid):
    """Consulta la API de la tienda de Steam para obtener detalles del juego."""
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}"
    try:
        response = requests.get(url)
        data = response.json()
        
        # Verificar si la respuesta es válida y exitosa
        if data and data[str(appid)]['success']:
            details = data[str(appid)]['data']
            
            # Extraer información estructurada
            info = {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'is_free': details.get('is_free', False),
                # Obtenemos el precio formateado (ej: "$9.99")
                'price_formatted': details.get('price_overview', {}).get('final_formatted', 'Gratis'),
                # Obtenemos puntuación de Metacritic (si existe)
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                # Obtenemos géneros como una cadena separada por comas
                'genres': ', '.join([g['description'] for g in details.get('genres', [])]),
                # Total de recomendaciones positivas
                'total_recommendations': details.get('recommendations', {}).get('total', 0)
            }
            return info
        else:
            return None
    except Exception as e:
        print(f"Error consultando {appid}: {e}")
        return None


#Creamos nuevo DataSet actualizado.

lista_datos_enriquecidos = []

# Iteramos sobre tus appids
for appid in df_juegos_maestro['appid']:
    info = obtener_info_juego(appid)
    if info:
        lista_datos_enriquecidos.append(info)
    # IMPORTANTE: Pausa para no saturar la API de Steam (Rate Limit)
    time.sleep(1) 

# Nuevo DF
df_juegos_maestro = pd.DataFrame(lista_datos_enriquecidos)

df_juegos_maestro.to_csv("./datasets/juegos_maestro.csv", index=False)


print("\nNuevo Dataset Maestro de Juegos generado:")
df_juegos_maestro.head()

In [ ]:
# Definimos los appids únicos
data_base = {
    'appid': steam_db_final["appid"].unique()
}

def obtener_info_juego(appid):
    # Consulta la API de la tienda de Steam para obtener detalles enriquecidos.
    # Añadimos &l=spanish para asegurar que los nombres y géneros vengan en español si están disponibles
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        if data and data[str(appid)]['success']:
            details = data[str(appid)]['data']
            
            # --- Lógica de Precio ---
            # Si is_free es True, el precio es 0. 
            # Si no, intentamos sacar el valor numérico (final) que viene en céntimos.
            is_free = details.get('is_free', False)
            if is_free:
                price = 0
            else:
                price_info = details.get('price_overview', {})
                # 'final' viene en céntimos (ej: 999 para 9.99€), lo dividimos por 100
                price = price_info.get('final', 0) / 100

            # --- Lógica de Género Único ---
            # Obtenemos la lista 
            genres_list = details.get('genres', [])
            
            def obtener_genero_prioritario(genres_list):
            # Definimos qué géneros queremos "cazar" primero por ser más descriptivos; ya que en Steam tienen varios géneros para cada juego. 
            # Estos se encuentran ordenados alfabéticamente por lo que no es correcto "quedarnos con el primero".
                
                prioridad = ["Rol", "Estrategia", "Simulación", "Deportes", "Carreras", "Aventura"]
                
                descripciones = [g['description'] for g in genres_list]
                
                # Si alguno de nuestra lista de prioridad está en el juego, lo elegimos
                for genero in prioridad:
                    if genero in descripciones:
                        return genero
                    
                # Si no hay ninguno de prioridad, devolvemos el primero que haya
                return descripciones[0] if descripciones else "Sin Género"
            
            main_genre = obtener_genero_prioritario(genres_list)

            # --- Lógica de Desarrollador ---
            # Los desarrolladores vienen en una lista, tomamos el primero
            devs_list = details.get('developers', [])
            developer = devs_list[0] if devs_list else "Desconocido"

            # Construcción del diccionario de retorno
            info = {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': developer,
                'price': price,
                'main_genre': main_genre,
                'is_free': is_free,
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                'total_recommendations': details.get('recommendations', {}).get('total', 0)
            }
            return info
        else:
            return None
    except Exception as e:
        print(f"Error consultando {appid}: {e}")
        return None

# 2. Proceso de extracción
lista_datos_enriquecidos = []

print(f"Iniciando extracción de {len(df_juegos_maestro)} juegos...")

for appid in df_juegos_maestro['appid']:
    info = obtener_info_juego(appid)
    if info:
        lista_datos_enriquecidos.append(info)
    
    # Pausa de 1.5 segundos para ser más conservadores con el Rate Limit de Steam
    time.sleep(1.5) 

# 3. Creación del DataFrame y guardado
df_juegos_maestro_act = pd.DataFrame(lista_datos_enriquecidos)

# Guardamos el resultado
df_juegos_maestro_act.to_csv("./datasets/juegos_maestro2.csv", index=False)

print("\nNuevo Dataset Maestro de Juegos generado con éxito:")
df_juegos_maestro_act.head()

# Final

In [ ]:
# 1. Preparación de los AppIDs únicos
appids_unicos = steam_db_final["appid"].unique()

# 2. Definición de la función de consulta (Lógica Enriquecida)
def obtener_info_juego(appid):
    """Consulta la API de Steam para obtener detalles enriquecidos en español."""
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        if data and data[str(appid)]['success']:
            details = data[str(appid)]['data']
            
            # --- Lógica de Precio ---
            is_free = details.get('is_free', False)
            if is_free:
                price = 0
            else:
                price_info = details.get('price_overview', {})
                # Convertimos céntimos a unidad estándar
                price = price_info.get('final', 0) / 100

            # --- Lógica de Género Único con Prioridad ---
            genres_list = details.get('genres', [])
            
            def obtener_genero_prioritario(genres_list):
                prioridad = ["Rol", "Estrategia", "Simulación", "Deportes", "Carreras", "Aventura"]
                descripciones = [g['description'] for g in genres_list]
                
                for genero in prioridad:
                    if genero in descripciones:
                        return genero
                return descripciones[0] if descripciones else "Sin Género"
            
            main_genre = obtener_genero_prioritario(genres_list)

            # --- Lógica de Desarrollador ---
            devs_list = details.get('developers', [])
            developer = devs_list[0] if devs_list else "Desconocido"

            # Construcción del diccionario de retorno
            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': developer,
                'price': price,
                'main_genre': main_genre,
                'is_free': is_free,
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                'total_recommendations': details.get('recommendations', {}).get('total', 0)
            }
        else:
            return None
    except Exception as e:
        print(f"Error consultando {appid}: {e}")
        return None

# 3. Proceso de extracción iterativo
lista_datos_enriquecidos = []
print(f"Iniciando extracción de {len(appids_unicos)} juegos...")

for appid in appids_unicos:
    info = obtener_info_juego(appid)
    if info:
        lista_datos_enriquecidos.append(info)
    
    # Pausa de seguridad para el Rate Limit de Steam
    time.sleep(1.5) 

# 4. Creación del DataFrame final y guardado
df_juegos_maestro = pd.DataFrame(lista_datos_enriquecidos)
df_juegos_maestro.to_csv("./datasets/juegos_maestro_final.csv", index=False)

print("\nNuevo Dataset Maestro de Juegos generado con éxito:")
df_juegos_maestro.head()

In [ ]:
df_juegos_maestro["main_genre"].value_counts()

In [ ]:
# Usamos Ollama para obtener la localización de la empresa desarrolladora.


# --- FUNCIÓN DE INTERACCIÓN ---
def obtener_pais_empresa(empresa):
    """
    Envía una solicitud a Ollama para obtener el país de una empresa.
    """
    if pd.isna(empresa) or empresa == '':
        return 'Desconocido'
    
    # Prompt optimizado para obtener respuestas cortas
    prompt = f"¿En qué país tiene su sede principal la empresa de videojuegos '{empresa}'? Responde solo con el nombre del país en español."
    
    try:
        response = ollama.chat(model=MODELO, messages=[
            {
                'role': 'user',
                'content': prompt,
            },
        ])
        return response['message']['content'].strip()
    except Exception as e:
        return f"Error: {e}"

# --- PROCESAMIENTO ---
print("Procesando las localizaciones de los estudios con Ollama...")

# Aplicar la función a la columna específica
df_juegos_maestro_act['country'] = df_juegos_maestro_act['developer'].apply(obtener_pais_empresa)

print("Procesamiento de localizaciones completado.")

# --- RESULTADO ---
# Mostramos las primeras filas para verificar
df_juegos_maestro_act[['developer', 'country']].head()

In [ ]:
def obtener_genero_limpio(juego, lista_generos_conocidos):
    """
    Envía una solicitud a Ollama para obtener un género genérico.
    """
    if pd.isna(juego) or juego == '':
        return 'Desconocido'
    
    # Esto le dice a Ollama qué palabras NO debe repetir o qué palabras mapear
    generos_str = ", ".join(list(set(lista_generos_conocidos)))

    # PROMPT MEJORADO Y RESTRINGIDO
    prompt = f"""
    Eres un experto clasificador de videojuegos.
    Objetivo: Define el género principal del juego '{juego}'.
    Reglas estrictas:
    1. Responde con exactamente UNA palabra.
    2. El género debe ser genérico (ej. "Acción", "RPG", "Estrategia"), NO específico.
    3. Usa español o inglés, el que sea más común para ese género.
    4. NO uses puntos finales.
    5. NO incluyas explicaciones ni contexto.
    6. Diferencia entre Acción y Shooters.
    7. IMPORTANTE: Consulta esta lista de géneros ya utilizados: [{generos_str}]. Si el juego encaja en uno de esa lista, usa la palabra exacta de la lista. Si es un género totalmente nuevo, propón uno nuevo pero genérico.
    """
    
    try:
        response = ollama.chat(model=MODELO, messages=[
            {'role': 'user', 'content': prompt},
        ])
        # Limpieza adicional para asegurar una sola palabra
        respuesta = response['message']['content'].strip().split('\n')[0]
        return respuesta
    except Exception as e:
        return "Error"

# --- PROCESAMIENTO ---
print("Procesando géneros con Ollama...")

# Para evitar repeticiones, llevamos una lista de lo que ya hemos categorizado
generos_ya_procesados = []

# Supongamos que tu columna de nombres se llama 'nombre_juego'
columna_nombres = 'name' 

# 1. Creamos la columna nueva vacía
df_juegos_maestro_act['genre'] = ''

# 2. Iteramos para ir alimentando la lista de conocidos (fila por fila)
for index, row in df_juegos_maestro_act.iterrows():
    juego = row[columna_nombres]
    
    # Pasamos la lista de lo que ya hemos ido respondiendo
    genero = obtener_genero_limpio(juego, generos_ya_procesados)
    
    df_juegos_maestro_act.at[index, 'genre'] = genero
    
    # Añadimos el nuevo género a nuestra lista de conocidos para la próxima iteración
    if genero not in generos_ya_procesados and genero != 'Error':
        generos_ya_procesados.append(genero)

print("Procesamiento de los géneros completado.")
# Mostramos resultados
df_juegos_maestro_act[[columna_nombres, 'genre']].head()

In [ ]:
df_juegos_maestro_act['genre'].value_counts()

- Añadir fecha lanzamiento
- Añadir idioma (para cruzarlo con paises)
- formatear precios (gratis - 0) (quitar simbolo €)
- Quedarnos con el género de la fila representativo del juego

In [46]:
# 1. Preparación de los AppIDs únicos
appids_unicos = steam_db_final["appid"].unique()

# --- FUNCIONES DE APOYO ---

def obtener_info_juego(appid):
    """Consulta la API de Steam para obtener detalles básicos."""
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    try:
        response = requests.get(url)
        data = response.json()
        if data and data[str(appid)]['success']:
            details = data[str(appid)]['data']
            is_free = details.get('is_free', False)
            price = 0 if is_free else (details.get('price_overview', {}).get('final', 0) / 100)
            devs_list = details.get('developers', [])
            
            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': devs_list[0] if devs_list else "Desconocido",
                'price': price,
                'is_free': is_free,
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                'total_recommendations': details.get('recommendations', {}).get('total', 0)
            }
    except Exception as e:
        print(f"Error en API Steam para {appid}: {e}")
    return None

def obtener_pais_empresa(empresa):
    """Obtiene el país de la empresa usando Ollama."""
    if pd.isna(empresa) or empresa == '' or empresa == 'Desconocido':
        return 'Desconocido'
    prompt = f"¿En qué país tiene su sede principal la empresa de videojuegos '{empresa}'? Responde solo con el nombre del país en español."
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content'].strip()
    except Exception:
        return "Error"

def obtener_genero_limpio(juego, lista_generos_conocidos):
    """Clasifica el género del juego usando Ollama."""
    if pd.isna(juego) or juego == '':
        return 'Desconocido'
    generos_str = ", ".join(list(set(lista_generos_conocidos)))
    prompt = f"""
    Eres un experto clasificador de videojuegos.
    Objetivo: Define el género principal del juego '{juego}'.
    Reglas estrictas:
    1. Responde con exactamente UNA palabra.
    2. El género debe ser genérico (ej. "Acción", "RPG", "Estrategia"), NO específico.
    3. Usa español o inglés, el que sea más común para ese género.
    4. NO uses puntos finales ni explicaciones.
    5. IMPORTANTE: Consulta esta lista: [{generos_str}]. Si encaja, usa esa palabra exacta.
    """
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content'].strip().split('\n')[0].replace(".", "")
    except Exception:
        return "Error"

# --- EJECUCIÓN DEL PROCESO ---

# PASO 1: Extracción de Steam (Bucle con Rate Limit)
lista_datos = []
print(f"Extrayendo datos de Steam ({len(appids_unicos)} juegos)...")
for appid in appids_unicos:
    info = obtener_info_juego(appid)
    if info:
        lista_datos.append(info)
    time.sleep(1.5) 

df_juegos_maestro_act = pd.DataFrame(lista_datos)

# PASO 2: Procesamiento de Países (Fuera del bucle de Steam)
print("Procesando localizaciones con Ollama...")
df_juegos_maestro_act['country'] = df_juegos_maestro_act['developer'].apply(obtener_pais_empresa)

# PASO 3: Procesamiento de Géneros (Iterativo para mantener memoria de géneros)
print("Procesando géneros con Ollama...")
generos_ya_procesados = []
df_juegos_maestro_act['genre'] = ''

for index, row in df_juegos_maestro_act.iterrows():
    genero = obtener_genero_limpio(row['name'], generos_ya_procesados)
    df_juegos_maestro_act.at[index, 'genre'] = genero
    if genero not in generos_ya_procesados and genero not in ['Error', 'Desconocido']:
        generos_ya_procesados.append(genero)

# PASO 4: Guardado y Visualización
df_juegos_maestro_act.to_csv("./datasets/juegos_maestro_ollama.csv", index=False)
print("\nProcesamiento completo. Dataset final:")
df_juegos_maestro_act.head()

Extrayendo datos de Steam (90 juegos)...
Procesando localizaciones con Ollama...
Procesando géneros con Ollama...

Procesamiento completo. Dataset final:


,appid,name,developer,price,is_free,metacritic_score,total_recommendations,country,genre
0,366240,GAROU: MARK OF THE WOLVES,SNK CORPORATION,9.99,False,0,897,Japón,RPG
1,730,Counter-Strike 2,Valve,0.00,True,0,4954122,Estados Unidos,Acción
2,1267910,Melvor Idle,Games by Malcs,9.75,False,0,14104,Reino Unido,RPG
3,1085660,Destiny 2,Bungie,0.00,True,83,133584,Estados Unidos,Acción
4,312660,Sniper Elite 4,Rebellion,59.99,False,78,53112,Reino Unido,Acción


In [49]:
df_juegos_maestro_act["country"].value_counts()

country
España             24
Estados Unidos     17
Reino Unido        14
Ucrania             8
Japón               5
Rusia               5
Rumania             2
Escocia             2
Singapur            2
República Checa     2
Dinamarca           2
Nueva Zelanda       1
Alemania            1
Canadá              1
Noruega             1
China               1
Corea del Sur       1
Irlanda             1
Name: count, dtype: int64